# BlackoutLab + ThreadCourt — live Ollama validation
Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. The notebook installs official Ollama, warms the model, measures actual runs, and renders both real interfaces inside Colab. Keep the runtime connected and idle to use BlackoutLab controls. Download evidence before the runtime expires.


In [ ]:
import subprocess, sys, pathlib, os, time, json, urllib.request, shutil
root=pathlib.Path('/content/incident-labs'); root.mkdir(exist_ok=True)
SOURCE_REF='main'
def run(args, **kw):
    result=subprocess.run(args,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,**kw)
    print(result.stdout,flush=True); result.check_returncode(); return result
run(['nvidia-smi'])
for name in ['BlackoutLab','ThreadCourt']:
    if not (root/name).exists():
        run(['git','clone','--branch',SOURCE_REF,'https://github.com/MinaIbrahim10/'+name+'.git',str(root/name)])
    else:
        run(['git','pull','--ff-only'],cwd=root/name)
    run(['git','rev-parse','HEAD'],cwd=root/name)
run([sys.executable,'-m','pip','install','-q','-e',str(root/'BlackoutLab')+'[test]','-e',str(root/'ThreadCourt')])
run([sys.executable,'-m','pytest','-q'],cwd=root/'BlackoutLab')
run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=root/'ThreadCourt')


## Start Ollama and warm the model
The current Linux installer requires `zstd`. Cold GPU initialization can exceed individual agent request timeouts, so warm the model before measuring experiments. Installation and model downloads use official Ollama endpoints.


In [ ]:
if not shutil.which('zstd'):
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','-qq','zstd'])
if not shutil.which('ollama'):
    urllib.request.urlretrieve('https://ollama.com/install.sh','/tmp/ollama-install.sh')
    run(['sh','/tmp/ollama-install.sh'])
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/version',timeout=2)
except Exception:
    ollama_process=subprocess.Popen(['ollama','serve'],env={**os.environ,'OLLAMA_HOST':'127.0.0.1:11434'},stdout=open(root/'ollama.log','w'),stderr=subprocess.STDOUT)
    for attempt in range(60):
        try: urllib.request.urlopen('http://127.0.0.1:11434/api/version',timeout=2); break
        except Exception: time.sleep(1)
    else: raise RuntimeError('Ollama failed to start; inspect ollama.log')
MODEL='gemma4:e4b'
run(['ollama','pull',MODEL])
os.environ.update(BLACKOUT_MODEL=MODEL,THREADCOURT_MODEL=MODEL,OLLAMA_BASE_URL='http://127.0.0.1:11434')
request=urllib.request.Request('http://127.0.0.1:11434/api/chat',data=json.dumps({'model':MODEL,'stream':False,'think':False,'messages':[{'role':'user','content':'Reply with OK'}],'options':{'num_ctx':8192,'num_predict':20}}).encode(),headers={'Content-Type':'application/json'})
with urllib.request.urlopen(request,timeout=600) as response:
    data=json.load(response); print('Warmup:',data.get('done_reason'),data.get('message',{}).get('content'))
run(['ollama','ps'])


## Real inference experiments
Single and multi modes use measured verification feedback, with up to 3 recovery rounds and no runbook fallback. Use `--max-rounds 1` to reproduce the earlier one-shot budget. Multi mode has additional analysis/verifier calls, so latency and success are not equal-cost comparisons. Retain unresolved runs as evidence.


In [ ]:
results=root/'results'; results.mkdir(exist_ok=True)
for mode in ['runbook','single','multi']:
    path=results/(mode+'.json')
    run([sys.executable,'-m','blackout','benchmark','--mode',mode,'--trials','1','--output',str(path)])
for scenario in ['lost-update','lock-inversion']:
    path=results/(scenario+'.json')
    result=subprocess.run([sys.executable,'-m','threadcourt',scenario,'--llm','--output',str(path)],text=True,capture_output=True,check=True)
    data=json.loads(path.read_text())
    print(scenario, 'before:',data['before']['verdict'],'after:',data['after']['verdict'])
    print([(agent['agent'],agent['status']) for agent in data['agents']])


## Use the dashboards inside Colab
Keep the runtime connected and idle while using BlackoutLab. Its notebook callback only permits lab API reads and validated run creation on loopback. No public port is required. ThreadCourt embeds the measured report with replay and a JSON download.


In [ ]:
sys.path[:0]=[str(root/'BlackoutLab'),str(root/'ThreadCourt')]
from IPython.display import display, HTML
from blackout.colab import display_lab
from threadcourt.dashboard import render_notebook
previous_server=globals().get('blackout_server')
if previous_server is not None and previous_server.poll() is None:
    previous_server.terminate(); previous_server.wait(timeout=10)
blackout_log=open('/content/blackout-server.log','a')
blackout_server=subprocess.Popen([sys.executable,'-m','blackout','serve','--port','8081'],cwd=root/'BlackoutLab',stdout=blackout_log,stderr=subprocess.STDOUT)
for attempt in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:8081/api/config'); break
    except Exception: time.sleep(1)
else: raise RuntimeError('Lab server did not start')
display_lab(8081)
# Change this to lock-inversion.json for the second measured scenario.
report=json.loads((results/'lost-update.json').read_text())
display(HTML(render_notebook(report)))


## Download measured evidence
This exports JSON reports, not Google credentials or browser sessions.


In [ ]:
import zipfile
from google.colab import files
bundle=root/'lab-evidence.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in results.glob('*.json'): archive.write(path,'results/'+path.name)
files.download(str(bundle))
